# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an example workflow for loading and exploring the A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library and the dataset's Croissant schema. You will learn to load, inspect, extract, and analyze data using structured schema `@id` references.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` and plotting libraries are installed
!pip install -q mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json['name']}: {metadata_json['description']}")
# Display basic info
print("\nFields in dataset metadata:", list(metadata_json.keys()))

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

Let's list all record sets, and for each, list their fields (with `@id` values and descriptions) as described in the Croissant schema.

In [ ]:
# List all record sets and their fields using @id references
ds = dataset

print('Available record sets with their @id:')
record_sets = ds.metadata.record_sets
for rs in record_sets:
    print(f"  - {rs['@id']}: {rs.get('name', '(no name)')}")
    if 'fields' in rs:
        print('    Fields:')
        for field in rs['fields']:
            fid = field.get('@id', '(no id)')
            fname = field.get('name', fid)
            fdesc = field.get('description', '')
            print(f"      {fid} ({fname}): {fdesc}")
    else:
        print('    Fields: <not listed in this record set>')

## 3. Data Extraction
Load data from one or multiple record sets into DataFrames for analysis. We will use the `@id` values from the previous step.

For demonstration, we'll take the main record set containing the primary survey data (for this dataset, the main `recordSet` holds the bulk of rows; if multiple, repeat as needed).

**Note:** Replace your target `record_set_id` or list with explicit `@id` values as printed above.

In [ ]:
# Extract data from main record set(s) into DataFrames
# Example: Suppose our main record set has @id 'cr:RecordSet/mental_health_survey_responses'
# Replace this with the correct value from the overview above.

# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in ds.metadata.record_sets]
print('Available record set @ids:', record_set_ids)

# For demonstration, we'll use the first record set
target_record_set_id = record_set_ids[0]

dataframes = {}
for rsid in record_set_ids:
    print(f"Loading records for record set {rsid} ...")
    records = list(ds.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)
    print(f"  Loaded {len(records)} records.")

# Check column names
print(f"\nColumns in record set {target_record_set_id}:")
print(dataframes[target_record_set_id].columns.tolist())

# Preview first few rows
dataframes[target_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Below, we will select a numeric field (`@id`), filter, normalize and group as appropriate.

**Tip:** Choose relevant field `@id` values, such as those related to GAD-7, PHQ-9, or PSQ scores.

For example purposes, let's use the field with `@id` `'cr:Field/gad7_total_score'` (replace with value according to your dataset fields overview, if different).

In [ ]:
import numpy as np

# Specify your numeric field and group field by their @id (as seen in the record set's fields list above)
numeric_field = 'cr:Field/gad7_total_score'  # Replace with actual @id if needed
group_field = 'cr:Field/gender'             # E.g., gender or another categorical field @id

df = dataframes[target_record_set_id]

if numeric_field in df.columns:
    threshold = 5
    filtered_df = df[df[numeric_field].astype(float) > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df[[numeric_field]].head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
    ) / filtered_df[numeric_field].astype(float).std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field if present
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'count'])
        print(f"\nGrouped data by {group_field}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field} not found in columns: {df.columns.tolist()}")

## 5. Visualization
Let's visualize the distribution of a key numeric field and the mean score by group/categorical variable (such as gender), using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the numeric field
if numeric_field in df.columns:
    plt.figure(figsize=(7,5))
    sns.histplot(df[numeric_field].dropna().astype(float), bins=10, kde=True)
    plt.title(f"Histogram of {numeric_field} values")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group (if available)
    if group_field in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field].astype(float))
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion

- We loaded the FAIR² Mental Health Survey Dataset using `mlcroissant` and explored its structure via Croissant schema `@id` references.
- We programmatically listed record sets and fields, making access robust to schema updates.
- Numeric and categorical data was extracted and processed, including filtering, normalization, grouping, and visualization.
- This approach ensures reproducible FAIR+AI workflows and precise references for future research.

**Next steps:**
- Extend analysis to other fields (e.g., PHQ-9, PSQ, education, age, location).
- Build predictive models or statistical analysis pipelines as needed!
